# XiangqiRL — PPO training on Kaggle

Trains the policy-value network with PPO against the random and heuristic baselines (Phase 4).

**Before running**
1. Upload `xiangqi-rl.zip` (built by `dist/xiangqi-rl.zip` in the repo) as a Kaggle **Dataset** and add it to this notebook (*Add Input*).
2. Notebook settings: **Accelerator = GPU T4** (CPU works too, only slower), **Internet = On** (needed once for `pip install gymnasium`).
3. *(Optional, to continue an earlier run)* upload `latest.pt` from a previous session as a dataset and set `RESUME_FROM` below.

Passing tests shows the code behaves as specified; it does **not** by itself show that the agent plays well. Judge that from the evaluation section at the end.

In [ ]:
import os, shutil, subprocess, sys, glob, time
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Locate and install the project

In [ ]:
# Find the uploaded project (a folder containing pyproject.toml + src/xiangqi_rl).
candidates = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
              if os.path.isdir(os.path.join(os.path.dirname(p), "src", "xiangqi_rl"))]
assert candidates, "Project not found under /kaggle/input - add the xiangqi-rl dataset as an input."
SRC = candidates[0]
PROJECT = "/kaggle/working/xiangqi-rl"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
shutil.copytree(SRC, PROJECT)
os.chdir(PROJECT)
print("project copied from", SRC, "to", PROJECT)

In [ ]:
# --no-deps keeps Kaggle's preinstalled (CUDA) torch; only the small extras are installed.
!pip install -q -e . --no-deps
!pip install -q gymnasium pyyaml
import xiangqi_rl, gymnasium, yaml
print("xiangqi_rl", xiangqi_rl.__version__, "| gymnasium", gymnasium.__version__)

## 2. Tests and smoke test (about a minute)

In [ ]:
!python -m pip install -q pytest
!python -m pytest -q -x tests

In [ ]:
!python scripts/train_ppo.py --config configs/ppo_baseline.yaml --smoke-test

## 3. Training configuration

Edit these values. Everything else comes from `configs/ppo_baseline.yaml`.
One iteration of the baseline config (2048 decisions) took about 25 s on a CPU-only
development machine; the game simulation is CPU-bound, so a T4 mainly speeds up the update.

In [ ]:
OUTPUT_DIR   = "/kaggle/working/ppo"     # checkpoints + CSV logs (kept as notebook output)
ITERATIONS   = 100                       # total rollout+update cycles
ROLLOUT_STEPS, MINIBATCH = 2048, 256
LEARNING_RATE = 3e-4                     # lower (e.g. 1e-4) if approx_kl / clip_fraction stay high
EVAL_INTERVAL, EVAL_GAMES = 10, 20       # evaluate every N iterations, games per opponent (even)
SEED = 0
RESUME_FROM = None                       # e.g. "/kaggle/input/xiangqi-ckpt/latest.pt"

overrides = {
    "total_iterations": ITERATIONS, "rollout_steps": ROLLOUT_STEPS, "minibatch_size": MINIBATCH,
    "learning_rate": LEARNING_RATE, "eval_interval": EVAL_INTERVAL, "eval_games": EVAL_GAMES,
    "seed": SEED, "output_dir": OUTPUT_DIR,
}
SET_ARGS = " ".join(f"--set {k}={v}" for k, v in overrides.items())
RESUME_ARGS = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
print(SET_ARGS, RESUME_ARGS)

## 4. Train

Re-running this cell with the same `OUTPUT_DIR` and `RESUME_FROM = OUTPUT_DIR + '/latest.pt'` continues training.

In [ ]:
!python scripts/train_ppo.py --config configs/ppo_baseline.yaml --device auto {SET_ARGS} {RESUME_ARGS}

## 5. Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(f"{OUTPUT_DIR}/train_log.csv")
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, ["episode_reward", "policy_loss", "value_loss",
                               "entropy", "approx_kl", "explained_variance"]):
    ax.plot(log["iteration"], log[col]); ax.set_title(col); ax.set_xlabel("iteration"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
log[["iteration", "decisions", "episode_reward", "wins", "draws", "losses",
     "env_seconds", "inference_seconds", "update_seconds"]].tail()

In [ ]:
ev = pd.read_csv(f"{OUTPUT_DIR}/eval_log.csv")
ev[["iteration", "opponent", "games", "wins", "draws", "losses", "win_rate", "score",
    "average_plies", "invalid_actions"]]

## 6. Final evaluation (separate games, many of them)

20 games per opponent is noisy. Use at least a few hundred before drawing conclusions,
and compare checkpoints with the same seed.

In [ ]:
!python scripts/evaluate_ppo.py --checkpoint {OUTPUT_DIR}/best.pt --games 200 --seed 12345 --device auto

## 7. Package the results

Download `ppo_run.zip` from the *Output* tab (or save a notebook version) and upload `latest.pt` as a dataset to resume later.

In [ ]:
shutil.make_archive("/kaggle/working/ppo_run", "zip", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))